# 01 — Loan-month panel (the base table)

**Plain English:** We build the single table that every later notebook reads:
**one row per loan per month**, tagged with its bucket and stage, the *next*
month's bucket and stage (so transitions are a lookup), and a few origination
attributes (state, balance, score). We cache it to `data/processed/panel.parquet`.

**One-line terms**
- **Loan-month panel** — the long table; rows = loans × the months they were observed.
- **Next-month state** — where the loan is one month later; the raw material for every transition matrix and roll rate.
- **Absorbing state** — Default and Prepaid close the loan's life; nothing rolls out of them.

The cached panel is **gitignored** — it is loan-level Freddie data and redistribution
is restricted. Only aggregated outputs are committed.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Build the panel and attach next-month transitions ----------------------
panel = m.build_panel()                 # one row per loan per month (+ bucket, stage, mob)
panel = m.add_transitions(panel)        # + next_bucket / next_stage (terminal events folded in)
print(f"{len(panel):,} loan-months  |  {panel.loan_seq.nunique():,} loans")

D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\src\monitor.py:194: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\src\monitor.py:194: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


8,850,130 loan-months  |  150,000 loans


In [3]:
# Join a few origination attributes used downstream ----------------------
# occupancy / loan_purpose / channel back the higher-risk-product concentration
# (APS 220 para 35) and third-party-originator monitoring (APS 220 para 39).
orig = pd.concat([m.load_orig(v) for v in m.VINTAGES], ignore_index=True)
keep = ["loan_seq", "vintage", "prop_state", "orig_upb", "credit_score", "dti", "ltv",
        "first_pmt_date", "occupancy", "loan_purpose", "channel"]
panel = panel.merge(orig[keep], on=["loan_seq", "vintage"], how="left")
panel.to_parquet(m.PROC_DIR / "panel.parquet", index=False)
print("cached ->", m.PROC_DIR / "panel.parquet")

cached -> D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\data\processed\panel.parquet


In [4]:
# One clean results table: shape of the base panel by vintage -------------
summ = (panel.groupby("vintage")
        .agg(loans=("loan_seq", "nunique"),
             loan_months=("loan_seq", "size"),
             avg_months_observed=("loan_age", lambda s: round(s.max(), 0)),
             current_share=("bucket", lambda s: round((s == "Current").mean(), 4)),
             ever_default_share=("bucket", lambda s: round(
                 panel.loc[s.index].assign(d=s.isin(["90+", "Default"]))
                      .groupby("loan_seq").d.max().mean(), 4)))
        .reset_index())
summ.to_csv(m.OUT_TABLES / "01_panel_summary.csv", index=False)
summ

,vintage,loans,loan_months,avg_months_observed,current_share,ever_default_share
0,2007,50000,3003932,225,0.9030,0.1647
1,2008,50000,2449679,212,0.9347,0.0930
2,2015,50000,3396519,128,0.9851,0.0406
